# Hệ thống RAG cho Chatbot Gọi món Nhà hàng: Thiết kế, Cài đặt và Đánh giá

**CMC Restaurant — QR AI Ordering** · Dịch vụ AI Python (FastAPI) · BM25 Okapi + LLM qua 9router

---

## Tóm tắt (Abstract)

Notebook này trình bày **toàn bộ kiến trúc và cơ sở lý thuyết** của dịch vụ AI trong hệ thống gọi món bằng mã QR:
một pipeline **RAG (Retrieval-Augmented Generation)** dùng bộ truy xuất từ vựng **BM25 Okapi** trên kho tri thức
Markdown tiếng Việt, kết hợp LLM (`gh/gpt-4o` qua gateway 9router) với **đầu ra JSON có cấu trúc**
và **nhiều lớp guardrail** chống bịa món/bịa giá.

Mọi đoạn mã trong notebook **chạy trực tiếp trên mã nguồn thật** của package `ai/app` — không có mã giả lập.
Các ô cần LLM/API hoặc thư viện ngoài (`sentence-transformers`, `pandas`, `matplotlib`) được bảo vệ bằng
kiểm tra khả dụng, do đó notebook **chạy được hoàn toàn offline** (các thí nghiệm cần thư viện sẽ tự bỏ qua
kèm hướng dẫn cài đặt).

**Nguyên tắc phương pháp luận:** mọi nhận định so sánh (BM25 vs embedding vs hybrid) trong notebook đều
được phát biểu dưới dạng **giả thuyết** và kiểm chứng bằng **thí nghiệm chạy được** trên bộ truy vấn có nhãn
(mục 6) — không có kết luận cảm tính.

## Mục lục

1. Tổng quan kiến trúc RAG
2. Cơ sở lý thuyết BM25
3. Xử lý tiếng Việt
4. Thiết kế Knowledge Base
5. Truy xuất thực nghiệm (định tính)
6. **Thí nghiệm so sánh BM25 vs Embedding vs Hybrid** (định lượng: hit@k, MRR, độ trễ, phân tích lỗi)
7. Prompt engineering & Output có cấu trúc
8. Guardrails & an toàn
9. Đánh giá (Evaluation) trên golden set
10. Độ trễ & tối ưu hiệu năng
11. Lộ trình nâng cấp

## 1. Tổng quan kiến trúc RAG

### 1.1. Pipeline tổng thể

Hệ thống chia thành ba tầng: frontend (React), backend nghiệp vụ (.NET) và dịch vụ AI (Python FastAPI).
Dịch vụ AI **không giữ trạng thái nghiệp vụ** — menu trực tiếp, giỏ hàng, đơn hàng đều do backend .NET
quản lý và gửi kèm trong mỗi request.

```mermaid
flowchart LR
    A[Khách hàng<br/>Frontend React] -->|"POST /api/chat/..."| B[.NET Backend<br/>ChatAssistantService]
    B -->|"POST /v1/chat<br/>(message, history, menu_items)"| C[Python AI Service<br/>FastAPI]

    subgraph C_inner [AiAssistantService — ai/app/services/assistant.py]
        C1[BM25Retriever<br/>top-k = 5] --> C2[Prompt Builder<br/>system policy + RAG context + menu]
        C2 --> C3[9router<br/>gh/gpt-4o]
        C3 --> C4[Output Parser<br/>JSON schema + validate menu_item_id]
        C0[Guardrails<br/>detect_guardrail_flags] --> C4
    end

    C --> C_inner
    C_inner -->|"ChatResponse JSON<br/>(content, sources, flags, actions)"| B
    B -->|validate nghiệp vụ| A

    KB[(Knowledge Base<br/>ai/knowledge-base/*.md)] --> C1
```

### 1.2. Vì sao chọn RAG thay vì fine-tuning?

Với miền hẹp là **thực đơn nhà hàng tiếng Việt**, RAG vượt trội fine-tuning trên cả ba trục:

| Tiêu chí | Fine-tuning | RAG (lựa chọn của hệ thống) |
| --- | --- | --- |
| **Chi phí** | Cần GPU, dữ liệu huấn luyện chất lượng cao, quy trình MLOps riêng | Chỉ cần soạn file Markdown; không train, không host model |
| **Độ tươi dữ liệu** | Menu/giá đổi → phải train lại; độ trễ cập nhật tính bằng ngày | Sửa file `.md` → nạp lại chunk; độ trễ cập nhật tính bằng giây |
| **Kiểm soát ảo giác** | Kiến thức "nướng" vào trọng số, khó truy vết nguồn | Mọi câu trả lời gắn với chunk truy xuất được (citation `source::title`), dễ audit |

### 1.3. Các quyết định kiến trúc chính

1. **Truy xuất từ vựng (BM25) thay vì embedding** — với kho tri thức nhỏ (~7 file Markdown) và nhu cầu khớp
   chính xác tên món, chúng tôi đặt **giả thuyết** rằng BM25 đủ tốt mà không cần model embedding hay vector DB.
   Giả thuyết này được **kiểm chứng định lượng** bằng thí nghiệm so sánh BM25/Embedding/Hybrid ở mục 6.
2. **LLM chỉ "diễn đạt", không "quyết định"** — mọi `menu_item_id` do LLM đề xuất đều bị đối chiếu lại với
   menu thật từ backend; món không tồn tại/hết hàng bị loại bỏ (mục 7, 8).
3. **Thang fallback nhiều bậc** — LLM hỏng không làm sập chat: hệ thống trả lời dựa trên RAG-only, và nếu
   không truy xuất được gì thì trả thông điệp tĩnh (mục 8).
4. **AI service tách khỏi backend .NET** — phần AI tiến hoá độc lập (Python, notebook, evaluation script)
   trong khi luật nghiệp vụ (tạo đơn, thanh toán) vẫn nằm trọn ở backend.

In [ ]:
# === Thiết lập môi trường: import package ai/app thật ===
# Notebook nằm ở ai/notebooks/ nên thêm ai/ vào sys.path để import "app.*"
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))

from app.rag.knowledge_base import load_markdown_knowledge_base
from app.rag.retriever import BM25Retriever, BM25_K1, BM25_B, TITLE_BOOST, TAG_BOOST

KB_PATH = Path("../knowledge-base")

chunks = load_markdown_knowledge_base(KB_PATH)
retriever = BM25Retriever(chunks)

print(f"Đã nạp knowledge base : {KB_PATH.resolve()}")
print(f"Số chunk              : {len(chunks)}")
print(f"Số file nguồn         : {len({c.source for c in chunks})}")
print(f"Siêu tham số BM25     : k1={BM25_K1}, b={BM25_B}, title_boost={TITLE_BOOST}, tag_boost={TAG_BOOST}")

## 2. Cơ sở lý thuyết BM25

### 2.1. Công thức Okapi BM25

Bộ truy xuất (`ai/app/rag/retriever.py`) cài đặt **Okapi BM25** (Robertson & Walker, 1994; Robertson & Zaragoza, 2009).
Với truy vấn $q$ gồm tập token $t \in q$ và tài liệu (chunk) $D$, điểm liên quan là:

$$
\mathrm{Score}(q, D) \;=\; \sum_{t \,\in\, q} \mathrm{IDF}(t) \cdot
\frac{\mathrm{tf}(t, D) \cdot (k_1 + 1)}
     {\mathrm{tf}(t, D) + k_1 \cdot \left(1 - b + b \cdot \dfrac{|D|}{\mathrm{avgdl}}\right)}
$$

trong đó ba thành phần đảm nhiệm ba vai trò riêng biệt:

**(a) IDF — độ hiếm của từ.** Cài đặt dùng biến thể IDF "trơn" (luôn không âm):

$$
\mathrm{IDF}(t) \;=\; \ln\!\left(\frac{N - \mathrm{df}(t) + 0.5}{\mathrm{df}(t) + 0.5} + 1\right)
$$

với $N$ là tổng số chunk và $\mathrm{df}(t)$ là số chunk chứa token $t$. Từ xuất hiện ở khắp nơi
(vd. "món", "nhà hàng" sau chuẩn hoá) có IDF thấp — đóng góp ít; từ đặc thù (vd. "hue" trong "bún bò Huế")
có IDF cao — quyết định thứ hạng.

**(b) Bão hoà tần suất từ — $k_1$.** Thành phần $\frac{\mathrm{tf}\,(k_1+1)}{\mathrm{tf} + k_1(\cdot)}$ là hàm
**tăng nhưng bão hoà**: lần xuất hiện thứ 10 của một từ đóng góp ít hơn nhiều so với lần đầu, tiệm cận
$k_1 + 1$. Điều này khắc phục nhược điểm của TF thô (tài liệu spam lặp từ khoá được điểm cao vô hạn).
Hệ thống dùng $k_1 = 1.5$ (mặc định chuẩn trong khoảng khuyến nghị $1.2$–$2.0$).

**(c) Chuẩn hoá độ dài — $b$.** Hệ số $1 - b + b\,\frac{|D|}{\mathrm{avgdl}}$ phạt chunk dài hơn trung bình
(chứa nhiều từ chỉ vì dài, không phải vì liên quan). $b = 0.75$ nghĩa là phạt 75% mức đầy đủ — mặc định chuẩn.

### 2.2. Mở rộng riêng của hệ thống: title/tag boosting

Trên nền BM25 chuẩn, retriever cộng thêm điểm thưởng **cộng tính** (additive):

$$
\mathrm{Score}_{\text{final}} = \mathrm{Score}_{\mathrm{BM25}}
+ 1.5 \cdot |q \cap \mathrm{title}(D)|
+ 1.0 \cdot |q \cap \mathrm{tags}(D)|
$$

Trực giác: tiêu đề chunk (heading Markdown, vd. "Phở bò đặc biệt") và tags (suy ra từ tên file, vd.
`allergy`, `dietary`) là tín hiệu chủ đề mạnh hơn nội dung thân — token truy vấn khớp tiêu đề đáng tin hơn
khớp giữa đoạn văn dài.

In [ ]:
# === Minh hoạ định lượng: IDF thực tế trên corpus + đường cong bão hoà TF ===
import math

# (a) IDF của một số token trên corpus THẬT (truy cập thống kê nội bộ của retriever)
N = retriever._num_docs
print(f"Corpus: N = {N} chunk, avgdl = {retriever._avg_doc_len:.1f} token/chunk\n")

print(f"{'Token':<12} {'df(t)':>6} {'IDF(t)':>8}   Nhận xét")
print("-" * 60)
for token in ["mon", "pho", "hue", "cay", "chay", "wifi", "vietqr"]:
    df = retriever._doc_freq.get(token, 0)
    idf = math.log((N - df + 0.5) / (df + 0.5) + 1.0)
    note = "phổ biến -> đóng góp thấp" if df > N * 0.3 else "đặc thù -> quyết định thứ hạng"
    print(f"{token:<12} {df:>6} {idf:>8.3f}   {note if df else 'không có trong corpus'}")

# (b) Bão hoà TF: đóng góp của một token theo số lần xuất hiện (doc dài trung bình, norm = 1)
print(f"\nBão hoà TF với k1 = {BM25_K1} (tiệm cận k1+1 = {BM25_K1 + 1}):")
print(f"{'tf':>4} {'đóng góp':>10} {'tăng thêm':>10}")
prev = 0.0
for tf in [1, 2, 3, 5, 10, 20]:
    contribution = tf * (BM25_K1 + 1) / (tf + BM25_K1)
    print(f"{tf:>4} {contribution:>10.3f} {contribution - prev:>+10.3f}")
    prev = contribution

### 2.3. Giả thuyết: BM25 phù hợp hơn embedding cho corpus tiếng Việt nhỏ này

Chúng tôi **không khẳng định trước** rằng BM25 tốt hơn embedding — đây là **giả thuyết cần kiểm chứng**,
với các lập luận tiên nghiệm (a priori) sau:

| Trục so sánh | Embedding + Vector DB | BM25 (giả thuyết) | Cách kiểm chứng |
| --- | --- | --- | --- |
| **Phụ thuộc hạ tầng** | Model embedding (~500MB+), vector store, pipeline re-index | Thuần stdlib Python (`math`, `re`, `unicodedata`) | Sự thật cấu trúc, không cần thí nghiệm |
| **Khớp tên món chính xác** | Có thể kéo "gỏi cuốn tôm thịt" về vùng ngữ nghĩa "món cuốn" chung | Token `goi`, `cuon`, `tom`, `thit` khớp đúng chunk, IDF thưởng từ hiếm | **Đo hit@1/MRR ở mục 6** + phân tích lỗi 6.6 |
| **Từ đồng nghĩa / paraphrase** | Kỳ vọng tốt hơn nhờ không gian ngữ nghĩa | Kỳ vọng yếu hơn (chỉ khớp token) | **Đo trên nhóm truy vấn paraphrase ở mục 6.6** |
| **Độ trễ truy vấn** | Phải encode truy vấn qua mạng neural | Chỉ tra bảng tần suất + số học | **Đo latency ms/truy vấn ở mục 6** |
| **Vận hành** | Re-embed khi sửa KB; version model; RAM cho model | Sửa `.md` là xong; index dựng lại tức thì | Sự thật cấu trúc |

Kết luận thực nghiệm (phương pháp nào hơn ở tiêu chí nào, kèm điều kiện thí nghiệm và giới hạn) được đưa ra
**duy nhất tại mục 6.5–6.7**, dựa trên bảng số liệu đo được.

### 2.4. Giới hạn đã biết của BM25 (trung thực về trade-off)

- **Từ đồng nghĩa (synonym gap):** "đồ uống giải khát" không khớp chunk chỉ viết "trà, cà phê" —
  BM25 không có khái niệm ngữ nghĩa. Giảm nhẹ hiện tại: soạn knowledge base có chủ đích, lặp các cách gọi
  phổ biến trong nội dung và tags. Mức độ ảnh hưởng thực tế được đo ở mục 6.6 (error analysis).
- **Diễn đạt lại (paraphrase):** "bụng yếu nên ăn gì" khó khớp "món dễ tiêu" nếu không trùng token nào.
- **Tách từ tiếng Việt:** tách theo khoảng trắng làm mất ngữ nghĩa từ ghép ("cà phê" thành 2 token) —
  phân tích chi tiết ở mục 3.
- **Không xếp hạng chéo tài liệu (no cross-doc semantics):** hai chunk cùng nói về một món không "biết" nhau.

Đây chính là động lực cho thí nghiệm **hybrid BM25 + embedding** ở mục 6.4 và lộ trình nâng cấp (mục 11).

## 3. Xử lý tiếng Việt

### 3.1. Chiến lược chuẩn hoá dấu (diacritic normalization)

Tiếng Việt có hệ dấu phong phú (thanh điệu + phụ âm/nguyên âm biến thể). Người dùng gõ chat thường
**không nhất quán**: "phở" / "pho" / "Phở", "bún bò Huế" / "bun bo hue". Nếu so khớp token nguyên bản,
"pho" sẽ trượt chunk viết "phở".

Pipeline chuẩn hoá trong `_tokenize_list` (`ai/app/rag/retriever.py`):

```text
"Bún bò Huế"
  → lower()                     "bún bò huế"
  → replace("đ", "d")           (NFKD không tách được "đ" vì không phải ký tự tổ hợp)
  → unicodedata.normalize(NFKD) tách chữ cái gốc + dấu tổ hợp: b,u,́,n, b,o,̀, h,u,ê,́ ...
  → lọc combining characters    "bun bo hue"
  → regex [a-z0-9]+             ["bun", "bo", "hue"]
```

Hệ quả: **truy vấn có dấu và không dấu cho cùng kết quả** — tính chất quan trọng với khách gõ trên điện thoại.

### 3.2. Trade-off của tokenization theo khoảng trắng

| Lựa chọn | Ưu điểm | Nhược điểm |
| --- | --- | --- |
| **Tách khoảng trắng + bỏ dấu** (hiện tại) | Không phụ thuộc thư viện; ổn định; nhanh | Mất nghĩa từ ghép: "cà phê" → `ca`, `phe`; mất phân biệt "bò" (thịt) vs "bó" sau khi bỏ dấu |
| Word segmentation (vd. `underthesea`, `pyvi`) | Giữ từ ghép "cà_phê", "gỏi_cuốn" | Thêm phụ thuộc nặng; chậm hơn; lỗi tách từ trên tên món riêng |
| Ký tự n-gram | Chịu lỗi chính tả tốt | Index lớn, điểm khó diễn giải |

Với corpus nhỏ mà tên món lặp lại nguyên văn trong knowledge base, tách khoảng trắng là điểm cân bằng hợp lý:
từ ghép bị tách nhưng **các token vẫn đồng xuất hiện trong cùng chunk**, nên BM25 cộng dồn điểm qua từng token
và chunk đúng vẫn thắng. Nhận định này được kiểm chứng định lượng ở mục 6 (nhóm truy vấn "tên món chính xác").

In [ ]:
# === Minh hoạ: chuẩn hoá + tokenize truy vấn tiếng Việt bằng hàm THẬT của retriever ===
from app.rag.retriever import _tokenize_list, _tokenize_set

samples = [
    "Món nào cay?",
    "gỏi cuốn tôm thịt",
    "Bún bò Huế còn không?",
    "bun bo hue con khong",          # cùng câu trên, gõ không dấu
    "Cà phê sữa đá",                  # "đ" -> "d", từ ghép bị tách
    "Tôi dị ứng hải sản",
]

for text in samples:
    print(f"{text!r:<35} -> {_tokenize_list(text)}")

# Kiểm chứng tính chất: có dấu và không dấu cho cùng tập token
assert _tokenize_set("Bún bò Huế còn không?") == _tokenize_set("bun bo hue con khong")
print("\n[OK] Truy vấn có dấu và không dấu chuẩn hoá về cùng tập token.")

## 4. Thiết kế Knowledge Base

### 4.1. Chiến lược chia chunk theo heading

Kho tri thức là các file Markdown trong `ai/knowledge-base/` (`menu.md`, `faq.md`, `ordering-policy.md`,
`combo-pairing.md`, `allergy-dietary.md`, `brand-voice.md`, `data-mining-insights.md`).
`load_markdown_knowledge_base` (`ai/app/rag/knowledge_base.py`) chia mỗi file thành chunk **tại mọi dòng
bắt đầu bằng `#`** (mọi cấp heading):

- Mỗi chunk mang: `source` (tên file), `title` (heading gần nhất), `content`, `tags` (tách từ tên file,
  vd. `allergy-dietary.md` → `("allergy", "dietary")`).
- Citation chuẩn: `source::title` — dùng để truy vết nguồn trong response và evaluation.

**Vì sao chia theo heading thay vì cửa sổ trượt (sliding window) cố định?** Tài liệu nhà hàng được soạn
có chủ đích: mỗi heading là **một đơn vị ngữ nghĩa trọn vẹn** (một câu hỏi FAQ, một nhóm món, một chính sách).
Chia theo heading giữ nguyên ranh giới ngữ nghĩa, tránh cắt đôi câu trả lời — đồng thời heading trở thành
`title` để boosting (mục 2.2). Cửa sổ cố định chỉ cần thiết khi tài liệu dài, phi cấu trúc.

### 4.2. Thống kê corpus (tính trực tiếp từ dữ liệu thật)

In [ ]:
# === Thống kê corpus: số chunk, token/chunk, phân bố theo file ===
from collections import defaultdict

per_file: dict[str, list[int]] = defaultdict(list)
for c in chunks:
    n_tokens = len(_tokenize_list(c.title + " " + c.content))
    per_file[c.source].append(n_tokens)

all_lens = [n for lens in per_file.values() for n in lens]

print(f"Tổng số chunk        : {len(chunks)}")
print(f"Token/chunk trung bình: {sum(all_lens) / len(all_lens):.1f}")
print(f"Token/chunk min–max   : {min(all_lens)}–{max(all_lens)}")
print(f"Kích thước từ vựng    : {len(retriever._doc_freq)} token duy nhất\n")

print(f"{'File':<28} {'Chunks':>6} {'Tổng token':>11} {'TB token/chunk':>15}")
print("-" * 64)
for source in sorted(per_file):
    lens = per_file[source]
    print(f"{source:<28} {len(lens):>6} {sum(lens):>11} {sum(lens)/len(lens):>15.1f}")

print("\nVí dụ 3 chunk đầu (citation):")
for c in chunks[:3]:
    print(f"  - {c.citation}  (tags={c.tags})")

## 5. Truy xuất thực nghiệm (định tính)

Chạy `BM25Retriever` thật trên các truy vấn tiếng Việt tiêu biểu, in top-3 chunk và điểm số.
Mục này mang tính **quan sát định tính** (đọc kết quả để hiểu hành vi xếp hạng);
đánh giá **định lượng** có nhãn đúng/sai nằm ở mục 6 và mục 9.

In [ ]:
# === Chạy BM25Retriever thật: top-3 kết quả cho 8 truy vấn mẫu ===
demo_queries = [
    "Món nào cay?",
    "gỏi cuốn tôm thịt",
    "Tôi bị dị ứng hải sản, nên tránh món nào?",
    "Nhà hàng mở cửa mấy giờ?",
    "Thanh toán bằng VietQR được không?",
    "Combo cho 2 người ăn trưa",
    "Món cay nên uống gì kèm?",
    "co mon chay khong",  # không dấu
]

for query in demo_queries:
    results = retriever.search(query, top_k=3)
    print(f"Truy vấn: {query!r}")
    if not results:
        print("   (không có kết quả)")
    for rank, item in enumerate(results, start=1):
        preview = item.chunk.content.replace("\n", " ")[:70]
        print(f"   #{rank} [{item.score:>7.4f}] {item.chunk.citation}")
        print(f"       {preview}...")
    print()

## 6. Thí nghiệm so sánh: BM25 vs Embedding vs Hybrid

Đây là phần **kiểm chứng định lượng** cho giả thuyết ở mục 2.3. Mọi kết luận trong mục 6.5–6.7
chỉ được rút ra từ bảng số liệu đo được, không từ cảm nhận.

### 6.1. Thiết kế thí nghiệm

**Bộ truy vấn có nhãn:** hợp nhất từ hai nguồn — (a) golden set `ai/evaluation/golden_questions.csv`
(các case có `expected_sources`), (b) bộ truy vấn bổ sung xây trong notebook, gắn nhãn nguồn đúng ở
**mức file** và phân nhóm (`group`) để phục vụ phân tích lỗi:

- `ten-mon` — tên món chính xác (kỳ vọng BM25 mạnh);
- `paraphrase` — diễn đạt lại/từ đồng nghĩa, cố ý **không dùng** từ khoá trong knowledge base (kỳ vọng embedding mạnh);
- `khong-dau` — truy vấn gõ không dấu;
- `faq`, `chinh-sach`, `golden` — câu hỏi vận hành/chính sách và case từ golden set.

**Định nghĩa "đúng":** chunk $D$ là liên quan với truy vấn $q$ nếu `D.source` $\in$ `expected_sources(q)`.
Nhãn ở mức file (không phải mức chunk) — đây là giới hạn đã biết của thí nghiệm, xem 6.7.

**Chỉ số đo:** với $r_q$ là hạng của chunk liên quan **đầu tiên** trong danh sách trả về:

$$
\mathrm{hit@}k = \frac{1}{|Q|}\sum_{q \in Q} \mathbb{1}[r_q \le k], \qquad
\mathrm{MRR} = \frac{1}{|Q|}\sum_{q \in Q} \frac{1}{r_q}
$$

(nếu không có chunk liên quan nào trong top-10 thì $1/r_q = 0$). Độ trễ đo bằng `time.perf_counter`
cho **toàn bộ bước truy xuất mỗi truy vấn** (với embedding bao gồm cả encode truy vấn — đúng chi phí
phải trả lúc chạy thật), lấy trung bình trên số lần lặp.

**Các phương pháp so sánh:**

1. **BM25** — chính `BM25Retriever` production (`ai/app/rag/retriever.py`), không chỉnh sửa.
2. **Embedding** — bi-encoder đa ngữ qua `sentence-transformers` + cosine similarity trên cùng bộ chunk.
   Model mặc định: `paraphrase-multilingual-MiniLM-L12-v2` (nhẹ, ~470MB); có thể đổi sang
   `bkai-foundation-models/vietnamese-bi-encoder` qua biến môi trường `NB_EMBEDDING_MODEL`.
3. **Hybrid RRF** — Reciprocal Rank Fusion trên hai danh sách xếp hạng.
4. **Hybrid weighted** — $\alpha \cdot \mathrm{BM25}_{\text{norm}} + (1-\alpha) \cdot \mathrm{cosine}_{\text{norm}}$
   với $\alpha \in \{0.3, 0.5, 0.7\}$ (chuẩn hoá min–max từng phía).

> Các ô embedding/hybrid được bọc `try/except ImportError`: nếu chưa cài `sentence-transformers`
> (`pip install sentence-transformers`) hoặc máy offline không tải được model, thí nghiệm **tự bỏ qua**
> và notebook vẫn chạy trọn vẹn với kết quả BM25.

In [ ]:
# === 6.2. Bộ truy vấn đánh giá có nhãn (golden set + bổ sung trong notebook) ===
import csv

GOLDEN_CSV = Path("../evaluation/golden_questions.csv")

eval_set: list[dict] = []

# (a) Nạp golden set thật (chỉ lấy case có expected_sources)
if GOLDEN_CSV.exists():
    with open(GOLDEN_CSV, encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            expected = {s.strip() for s in row["expected_sources"].split(";") if s.strip()}
            if expected:
                eval_set.append({"query": row["user_question"], "expected": expected, "group": "golden"})
    print(f"Nạp {len(eval_set)} case có nhãn nguồn từ {GOLDEN_CSV}")
else:
    print(f"Không tìm thấy {GOLDEN_CSV} — chỉ dùng bộ bổ sung trong notebook.")

# (b) Bộ bổ sung — nhãn gán thủ công ở mức file, dựa trên nội dung thật của knowledge base
EXTRA_CASES = [
    # --- Tên món chính xác (kỳ vọng lexical mạnh) ---
    ("gỏi cuốn tôm thịt",                     {"menu.md", "allergy-dietary.md"},      "ten-mon"),
    ("bún bò huế",                            {"menu.md", "combo-pairing.md"},        "ten-mon"),
    ("cơm gà xối mỡ",                         {"menu.md", "combo-pairing.md"},        "ten-mon"),
    ("chả giò hải sản",                       {"menu.md", "allergy-dietary.md"},      "ten-mon"),
    ("trà đào cam sả",                        {"menu.md", "combo-pairing.md"},        "ten-mon"),
    ("chè khúc bạch",                         {"menu.md", "combo-pairing.md"},        "ten-mon"),
    ("bánh flan caramel còn không",           {"menu.md"},                            "ten-mon"),
    # --- Không dấu (kiểm chứng chuẩn hoá NFKD) ---
    ("com suon nuong",                        {"menu.md", "combo-pairing.md"},        "khong-dau"),
    ("ca phe sua da",                         {"menu.md", "combo-pairing.md"},        "khong-dau"),
    ("goi cuon tom thit",                     {"menu.md", "allergy-dietary.md"},      "khong-dau"),
    # --- Paraphrase / từ đồng nghĩa: cố ý tránh từ khoá trong KB (kỳ vọng embedding mạnh) ---
    ("tôi không ăn được đồ biển",             {"allergy-dietary.md"},                 "paraphrase"),
    ("có gì cho người đang giảm cân không",   {"allergy-dietary.md"},                 "paraphrase"),
    ("đi hai đứa thì kêu gì bây giờ",         {"combo-pairing.md"},                   "paraphrase"),
    ("quán có mạng không",                    {"faq.md"},                             "paraphrase"),
    ("quán còn hoạt động lúc mấy giờ",        {"faq.md"},                             "paraphrase"),
    ("ăn xong khát nước quá có gì uống",      {"menu.md", "combo-pairing.md"},        "paraphrase"),
    # --- FAQ / chính sách vận hành ---
    ("mật khẩu wifi là gì",                   {"faq.md"},                             "faq"),
    ("để xe ô tô ở đâu",                      {"faq.md"},                             "faq"),
    ("trả bằng thẻ hay tiền mặt",             {"faq.md"},                             "faq"),
    ("huỷ đơn đã gửi được không",             {"faq.md", "ordering-policy.md"},       "chinh-sach"),
    ("AI có tự đặt đơn cho tôi không",        {"ordering-policy.md", "faq.md"},       "chinh-sach"),
    ("món hết hàng thì xử lý thế nào",        {"ordering-policy.md"},                 "chinh-sach"),
]
for query, expected, group in EXTRA_CASES:
    eval_set.append({"query": query, "expected": expected, "group": group})

from collections import Counter
print(f"\nTổng cộng: {len(eval_set)} truy vấn có nhãn")
print("Phân bố theo nhóm:", dict(Counter(c['group'] for c in eval_set)))
assert len(eval_set) >= 20, "Bộ đánh giá phải có tối thiểu 20 truy vấn"